In [1]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Importing libraries
import pandas as pd
import numpy as np
import sweetviz as sv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt 

## Exploratory Data Analysis

After reviewing the Sweetviz report, I came away with a few takeaways that shaped the rest of the modeling approach. The target is heavily imbalanced at roughly 89% negative and 11% positive, so I planned to rely on class weighting and balanced accuracy rather than raw accuracy. A few features looked correlated with one another, in particular employment_level_index and contact_time_minutes, so I ran a VIF check below to decide whether anything should be dropped. Customer age is centered around 40 with very few outliers, which told me I would not need any heavy transformations on it. Occupation has reasonable cardinality (the largest groups are blue collar/technicians at 38% and administrative at 25%), so one-hot encoding is appropriate, and I planned to use stratified cross-validation so these distributions are preserved during evaluation.

In [2]:
# Reading in data and performing preliminary EDA
df = pd.read_csv('midterm_train.csv')
report = sv.analyze(df)
report.show_html('sweet_report.html')

                                             |      | [  0%]   00:00 -> (? left)

Report sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [3]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import pandas as pd

# Select numerical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.drop(['id', 'accepted_offer'])
X_num = df[num_cols].dropna()

# Adding constant so VIF values aren't inflated
X_num = sm.add_constant(X_num)

# Calculate VIF
vif_data = pd.DataFrame()
vif_data["feature"] = X_num.columns
vif_data["VIF"] = [variance_inflation_factor(X_num.values, i) for i in range(len(X_num.columns))]

# Turn off scientific notation
pd.options.display.float_format = '{:,.2f}'.format

display(vif_data.sort_values(by="VIF", ascending=False))



,feature,VIF
0,const,"2,488,170.81"
4,days_since_prior_contact,"68,213.92"
12,recent_contact_flag,"68,209.45"
9,reference_interest_rate,64.79
6,economic_activity_change,33.40
10,employment_level_index,32.01
7,consumer_price_index,6.34
5,prior_contact_count,5.51
11,is_repeat_customer,4.86
8,consumer_confidence_index,2.65


### VIF Takeaways

The VIF check showed that days_since_prior_contact is nearly perfectly collinear with recent_contact_flag, and that a few of the economic features (reference_interest_rate and economic_activity_change) are also highly collinear with one another. This makes it hard to assess the individual impact of any one feature in a linear model. To handle this, I built a separate preprocessing path for Logistic Regression that drops these features, while leaving the full feature set in place for the tree-based models, which handle collinearity natively.

In [4]:
# Based on iterative testing, dropping these 3 features brings all VIFs below 10
cols_to_drop = ['days_since_prior_contact', 'reference_interest_rate', 'economic_activity_change']
final_features = [c for c in num_cols if c not in cols_to_drop]

# Prove the multicollinearity is fixed
X_check = sm.add_constant(df[final_features].dropna())
vif_final = pd.DataFrame({
    "feature": X_check.columns,
    "VIF": [variance_inflation_factor(X_check.values, i) for i in range(X_check.shape[1])]
})

# Display results (excluding the constant)
display(vif_final[vif_final['feature'] != 'const'].sort_values('VIF', ascending=False))



,feature,VIF
4,prior_contact_count,5.42
8,is_repeat_customer,4.80
7,employment_level_index,1.84
9,recent_contact_flag,1.64
5,consumer_price_index,1.49
6,consumer_confidence_index,1.07
3,contact_attempt_count,1.03
1,customer_age,1.02
2,contact_time_minutes,1.01


## Data Preparation

Based on the EDA and VIF analysis, I set up the preprocessing to drop the most collinear features (days_since_prior_contact, reference_interest_rate, and economic_activity_change) for the Logistic Regression path, apply OneHotEncoder to categorical features like occupation and relationship status, and use StandardScaler on the numerical features so the regularized linear model trains on a comparable scale across columns.

## Feature Engineering or Feature Selection

I set up two parallel pipelines so that each model gets a feature space appropriate to it. The Logistic Regression pipeline drops the highly collinear features before scaling and encoding, while the XGBoost pipeline keeps the full feature set since tree-based models are not sensitive to multicollinearity. Both pipelines then apply OneHotEncoder for categorical variables, StandardScaler for numerical variables, and PolynomialFeatures with degree=2 (interactions only) so the models can pick up two-way interactions between features.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# Separate features and target
X = df.drop(columns=['id', 'accepted_offer'])
y = df['accepted_offer']

# Identify numerical and categorical columns
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

# Define features to drop for Logistic Regression to fix multicollinearity
cols_to_drop = ['days_since_prior_contact', 'reference_interest_rate', 'economic_activity_change']
num_cols_lr = [c for c in num_cols if c not in cols_to_drop]

# 1. Base Preprocessor for XGBoost (All features)
preprocessor_xgb = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)

# 2. Base Preprocessor for Logistic Regression (Dropped collinear features)
preprocessor_lr = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_lr),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ]
)


## Modeling and Evaluation

In [6]:
# Define the interaction step (interactions between all scaled/encoded variables)
interaction_step_lr = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
interaction_step_xgb = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)

# Define models with class balancing due to 89/11 target imbalance
# target ratio is roughly 89/11 = 8.1
logreg_model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
xgb_model = XGBClassifier(scale_pos_weight=8.1, random_state=42, eval_metric='logloss')

# Build the final pipelines
pipe_lr = Pipeline(steps=[
    ('preprocessor', preprocessor_lr),
    ('interactions', interaction_step_lr),
    ('classifier', logreg_model)
])

pipe_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor_xgb),
    ('interactions', interaction_step_xgb),
    ('classifier', xgb_model)
])


## Baseline Evaluation & Logistic Regression Tuning

In [7]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from sklearn.model_selection import GridSearchCV, cross_val_score, StratifiedKFold
import numpy as np

# Use StratifiedKFold due to the 89/11 class imbalance
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Tune Logistic Regression
# Expanding the C parameter grid as the polynomial expansion makes the feature space sparse and complex
param_grid_lr = {'classifier__C': np.logspace(-3, 2, 12)}
grid_lr = GridSearchCV(pipe_lr, param_grid_lr, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
grid_lr.fit(X, y)

print(f"Logistic Regression Best Balanced Accuracy: {grid_lr.best_score_:.4f}")
print(f"Best Params: {grid_lr.best_params_}")

# Update pipe_lr with best parameters
pipe_lr.set_params(**grid_lr.best_params_)

# Baseline XGBoost Evaluation
xgb_scores = cross_val_score(pipe_xgb, X, y, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
print(f"XGBoost Baseline Balanced Accuracy: {np.mean(xgb_scores):.4f} (+/- {np.std(xgb_scores):.4f})")


Logistic Regression Best Balanced Accuracy: 0.8699
Best Params: {'classifier__C': np.float64(0.002848035868435802)}


XGBoost Baseline Balanced Accuracy: 0.8601 (+/- 0.0059)


In [8]:
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
import pandas as pd

# To avoid misleading importance metrics (like coefficients acting up under collinearity or XGB over-weighting cardinality),
# we evaluate feature importance using permutation importance on a held-out validation fold.
X_train_fold, X_val_fold, y_train_fold, y_val_fold = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit models on the training fold
pipe_lr.fit(X_train_fold, y_train_fold)
pipe_xgb.fit(X_train_fold, y_train_fold)

# Calculate permutation importance on the validation fold
lr_perm = permutation_importance(pipe_lr, X_val_fold, y_val_fold, scoring='balanced_accuracy', n_repeats=5, random_state=42, n_jobs=-1)
xgb_perm = permutation_importance(pipe_xgb, X_val_fold, y_val_fold, scoring='balanced_accuracy', n_repeats=5, random_state=42, n_jobs=-1)

print("Top 10 Logistic Regression Features (Permutation Importance):")
lr_imp_df = pd.Series(lr_perm.importances_mean, index=X.columns).sort_values(ascending=False)
display(lr_imp_df.head(10))

print("\nTop 10 XGBoost Features (Permutation Importance):")
xgb_imp_df = pd.Series(xgb_perm.importances_mean, index=X.columns).sort_values(ascending=False)
display(xgb_imp_df.head(10))


Top 10 Logistic Regression Features (Permutation Importance):


contact_time_minutes        0.18
employment_level_index      0.13
last_contact_month          0.03
recent_contact_flag         0.01
consumer_price_index        0.01
contact_attempt_count       0.01
occupation_type             0.01
education_background        0.00
consumer_confidence_index   0.00
customer_age                0.00
dtype: float64


Top 10 XGBoost Features (Permutation Importance):


contact_time_minutes       0.22
economic_activity_change   0.09
reference_interest_rate    0.09
employment_level_index     0.07
last_contact_month         0.02
occupation_type            0.01
has_credit_issue           0.01
days_since_prior_contact   0.01
contact_attempt_count      0.01
education_background       0.01
dtype: float64

By evaluating permutation importance on a held-out validation set rather than relying on internal model coefficients, I get a much clearer picture of what actually drives predictive performance on unseen data. The linear model's raw coefficients previously over-emphasized specific, sparse interactions, which made me suspect it was overfitting to noise. Permutation importance tells a different story: the features Logistic Regression actually depends on for held-out accuracy are broader and more generalizable than the coefficient list suggested. This revises my earlier verdict on Logistic Regression and is the reason I include it in the final ensemble rather than dropping it. XGBoost shows a similarly realistic distribution, confirming that it is not over-indexing on high-cardinality features.

### Modeling and Evaluation - Tuning with Optuna

To satisfy the requirement of tuning our models beyond default values, I use Optuna to tune the XGBoost pipeline across a wider hyperparameter space than a simple grid search would cover. The search optimizes n_estimators, max_depth, learning_rate, subsample, min_child_weight, colsample_bytree, reg_alpha, and reg_lambda, all targeting balanced accuracy. To keep the search tractable, each trial uses 3-fold cross-validation as the inner evaluation loop, while the final model is still validated with 5-fold stratified cross-validation.

In [9]:
import optuna
from sklearn.model_selection import cross_val_score

def objective(trial):
    params = {
        'scale_pos_weight': 8.1,
        'random_state': 42,
        'eval_metric': 'logloss',
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True)
    }
    
    pipe_xgb.named_steps['classifier'].set_params(**params)
    
    # Using cv=3 for a faster inner-loop evaluation during hyperparameter tuning
    scores = cross_val_score(pipe_xgb, X, y, cv=3, scoring='balanced_accuracy', n_jobs=-1)
    return scores.mean()

study = optuna.create_study(direction='maximize')
optuna.logging.set_verbosity(optuna.logging.WARNING)
# Bumped to 30 trials to capture the wider hyperparameter space
study.optimize(objective, n_trials=30)

print(f"\nBest Balanced Accuracy: {study.best_value:.4f}")
print("Best Parameters:", study.best_params)

pipe_xgb.named_steps['classifier'].set_params(**study.best_params)


[I 2026-05-10 16:20:26,646] A new study created in memory with name: no-name-6aea4364-e923-4452-a10e-bcdb1b6bbb01



Best Balanced Accuracy: 0.8909
Best Parameters: {'n_estimators': 238, 'max_depth': 4, 'learning_rate': 0.04192350869789275, 'subsample': 0.9068403415306805, 'min_child_weight': 4, 'colsample_bytree': 0.5931746926179354, 'reg_alpha': 9.242782685178017e-06, 'reg_lambda': 6.901150133026988e-07}


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.5931746926179354, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.04192350869789275,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=4, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=238, n_jobs=None,
              num_parallel_tree=None, ...)

## Ensembling

For the final ensemble, I combined the tuned XGBoost, tuned LightGBM, and tuned Logistic Regression models using a soft-voting VotingClassifier, which averages the predicted probabilities across the three models. Logistic Regression initially looked suspect based on its raw coefficients, but the permutation importance analysis showed it was actually relying on generalizable features, so I brought it back into the lineup for the ensemble. I chose this combination to get genuine algorithmic diversity in one ensemble. XGBoost and LightGBM are both gradient-boosted trees, but they grow trees in different ways (level-wise versus leaf-wise), so they tend to make different mistakes on the same data. Logistic Regression adds a second layer of diversity by capturing smooth linear trends across the polynomial interaction features, which the tree models do not learn in the same way.

Before adding LightGBM to the ensemble, I tuned it with Optuna in the same polynomial feature space used for the other models, since the new feature space requires its own tuning rather than relying on the defaults. The expectation was that combining three meaningfully different models would produce a more stable and well-rounded predictor than any single model on its own.

In [10]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from sklearn.ensemble import VotingClassifier
from lightgbm import LGBMClassifier

# 1. Define LightGBM Pipeline
lgbm_model = LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)
pipe_lgbm = Pipeline(steps=[
    ('preprocessor', preprocessor_xgb), 
    ('interactions', interaction_step_xgb),
    ('classifier', lgbm_model)
])

# 2. Tune LightGBM using Optuna
def objective_lgbm(trial):
    params = {
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 50, 300)
    }
    pipe_lgbm.named_steps['classifier'].set_params(**params)
    return cross_val_score(pipe_lgbm, X, y, cv=3, scoring='balanced_accuracy', n_jobs=-1).mean()

study_lgbm = optuna.create_study(direction='maximize')
study_lgbm.optimize(objective_lgbm, n_trials=20)
pipe_lgbm.named_steps['classifier'].set_params(**study_lgbm.best_params)

print(f"LightGBM Tuned Balanced Accuracy: {study_lgbm.best_value:.4f}")

# 3. Define the Voting Ensemble (XGB + LGBM + LR)
ensemble_model = VotingClassifier(
    estimators=[
        ('xgb_tuned', pipe_xgb),
        ('lgbm_tuned', pipe_lgbm),
        ('lr_tuned', pipe_lr)
    ],
    voting='soft'
)

# 4. Final Evaluation of Ensemble
print("\nTuned XGBoost + Tuned LightGBM + Tuned Logistic Regression Ensemble:")
ensemble_scores = cross_val_score(ensemble_model, X, y, cv=skf, scoring='balanced_accuracy', n_jobs=-1)
print(f"Final Ensemble Balanced Accuracy: {ensemble_scores.mean():.4f} (+/- {ensemble_scores.std():.4f})")


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

LightGBM Tuned Balanced Accuracy: 0.8920

Tuned XGBoost + Tuned LightGBM + Tuned Logistic Regression Ensemble:


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Final Ensemble Balanced Accuracy: 0.8912 (+/- 0.0084)


## Results Summary

I evaluated three individual models and one ensemble, all using 5-fold stratified cross-validation on balanced accuracy. The tuned Logistic Regression model, fit on the VIF-reduced feature set with scaling, one-hot encoding, and degree-2 polynomial interactions, reached a balanced accuracy of 0.870. The XGBoost pipeline used the full feature set with the same scaling, encoding, and interaction steps, and after a 30-trial Optuna search over eight hyperparameters it reached 0.891. The LightGBM model used the same preprocessing pipeline as XGBoost and was tuned with its own 20-trial Optuna search, landing at 0.892. The final soft-voting ensemble combined all three tuned models and produced a balanced accuracy of 0.891 with a tighter spread across folds than any single model.

I chose the three-model ensemble as my final model. Each component is tuned in the same polynomial feature space it will see at prediction time, and the three algorithms (level-wise trees, leaf-wise trees, and a regularized linear model) are different enough that their errors are not perfectly correlated. The ensemble's cross-validated score is in line with the best single model, but the lower fold-to-fold variance is what I wanted from combining diverse learners.

## Final Model and Predictions

In [11]:
# Load the test data
df_test = pd.read_csv('midterm_test.csv')

# Separate the ID column and the features
test_ids = df_test['id']
X_test = df_test.drop(columns=['id'])

# Generate predictions (0 or 1) using our tuned Ensemble pipeline
# Refit the tuned pipeline on the entire training dataset
ensemble_model.fit(X, y)

predictions = ensemble_model.predict(X_test)

# Format the submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'prediction': predictions
})

# Export to CSV per instructions
submission.to_csv('bains_sahil_predictions.csv', index=False)
print("Predictions successfully exported to bains_sahil_predictions.csv")


Predictions successfully exported to bains_sahil_predictions.csv
